# Paper Model Reproduction: Stacking with Gradient Boosting Meta Learner

Base models: Random Forest, LSTM, Deep Neural Network, Extra Trees

Setup: hourly panel windows, hypothesis-test differencing, history windows 16 and 28, forecast horizon 4

In [1]:
import importlib.util
import os
import random
import subprocess
import sys
import time
import warnings

required_packages = {
    "numpy": "numpy",
    "pandas": "pandas",
    "sklearn": "scikit-learn",
    "statsmodels": "statsmodels",
    "tensorflow": "tensorflow"
}

for module_name, package_name in required_packages.items():
    if importlib.util.find_spec(module_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.ensemble import ExtraTreesRegressor, GradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import ParameterSampler
from sklearn.multioutput import MultiOutputRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from statsmodels.tsa.stattools import adfuller
from tensorflow.keras import Sequential
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Dense, Dropout, LSTM

np.random.seed(42)
random.seed(42)
tf.random.set_seed(42)

2026-04-21 18:34:44.289181: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776796484.525728      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776796484.591147      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776796485.106065      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776796485.106105      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776796485.106108      55 computation_placer.cc:177] computation placer alr

In [2]:
candidate_paths = [
    "/kaggle/input/total-consumption-data/total_consumption_data.csv",
    "/kaggle/input/datasets/rahulsarav/total-consumption-data/total_consumption_data.csv",
    "/kaggle/input/total_consumption_data.csv",
    "total_consumption_data.csv"
]

data_path = None
for p in candidate_paths:
    if os.path.exists(p):
        data_path = p
        break

if data_path is None:
    raise FileNotFoundError("total_consumption_data.csv was not found in expected Kaggle/local paths")

df = pd.read_csv(data_path)
df.columns = [c.strip() for c in df.columns]

if np.issubdtype(df["datetime"].dtype, np.number):
    df["datetime"] = pd.to_datetime(df["datetime"], unit="s", errors="coerce")
else:
    df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")

for col in df.columns:
    if col not in ["datetime", "meter_id"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

target_col = "apower_ph1" if "apower_ph1" in df.columns else "apower"
df = df.dropna(subset=["datetime", "meter_id", target_col]).copy()
df = df.sort_values(["meter_id", "datetime"]).reset_index(drop=True)

num_cols = [c for c in df.columns if c not in ["datetime", "meter_id"]]
df[num_cols] = df.groupby("meter_id")[num_cols].ffill()
df[num_cols] = df.groupby("meter_id")[num_cols].bfill()

df["hour"] = df["datetime"].dt.floor("h")
hourly = df.groupby(["meter_id", "hour"], as_index=False)[target_col].mean()
meter_rank = hourly.groupby("meter_id").size().sort_values(ascending=False)
selected_meters = meter_rank.head(40).index
hourly = hourly[hourly["meter_id"].isin(selected_meters)].copy()
hourly = hourly.sort_values(["meter_id", "hour"]).reset_index(drop=True)

print("Data path:", data_path)
print("Target column:", target_col)
print("Meters used:", hourly["meter_id"].nunique())
print("Hourly rows:", len(hourly))
hourly.head()

Data path: /kaggle/input/datasets/rahulsarav/total-consumption-data/total_consumption_data.csv
Target column: apower_ph1
Meters used: 6
Hourly rows: 2420


,meter_id,hour,apower_ph1
0,00124B0018D6F607,2019-08-27 22:00:00,445.683333
1,00124B0018D6F607,2019-08-27 23:00:00,647.000000
2,00124B0018D6F607,2019-08-28 00:00:00,549.716667
3,00124B0018D6F607,2019-08-28 01:00:00,517.866667
4,00124B0018D6F607,2019-08-28 02:00:00,418.224138


In [3]:
def rmse_flat(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(np.asarray(y_true).reshape(-1), np.asarray(y_pred).reshape(-1))))

def choose_diff_order(series, max_d=1, alpha=0.05):
    p_values = {}
    for d in range(max_d + 1):
        s = series.diff(d).dropna() if d > 0 else series.dropna()
        if len(s) < 50:
            continue
        pval = adfuller(s, autolag="AIC")[1]
        p_values[d] = float(pval)
        if pval < alpha:
            return d, p_values
    if len(p_values) == 0:
        return 0, p_values
    return max(p_values.keys()), p_values

def build_meter_windows(values, window, horizon, d):
    values = np.asarray(values, dtype=np.float32)
    diff_values = pd.Series(values).diff(d).to_numpy(dtype=np.float32) if d > 0 else values.copy()
    x_buf = []
    y_buf = []
    base_buf = []
    end_buf = []
    n = len(values)
    start = max(window, d)
    for t in range(start, n - horizon + 1):
        x = diff_values[t - window:t]
        y = diff_values[t:t + horizon]
        if np.isnan(x).any() or np.isnan(y).any():
            continue
        x_buf.append(x)
        y_buf.append(y)
        base_buf.append(values[t - 1])
        end_buf.append(t + horizon - 1)
    if len(x_buf) == 0:
        return None
    return (
        np.asarray(x_buf, dtype=np.float32),
        np.asarray(y_buf, dtype=np.float32),
        np.asarray(base_buf, dtype=np.float32),
        np.asarray(end_buf, dtype=np.int32)
    )

def build_panel_windows(frame, target, window, horizon, d):
    all_x = []
    all_y = []
    all_base = []
    all_split = []
    for meter_id, g in frame.groupby("meter_id"):
        g = g.sort_values("hour").reset_index(drop=True)
        if len(g) < window + horizon + 24:
            continue
        built = build_meter_windows(g[target].values, window, horizon, d)
        if built is None:
            continue
        x_m, y_m, base_m, end_m = built
        n = len(g)
        train_cut = int(n * 0.70) - 1
        val_cut = int(n * 0.85) - 1
        split_m = np.where(end_m <= train_cut, "train", np.where(end_m <= val_cut, "val", "test"))
        all_x.append(x_m)
        all_y.append(y_m)
        all_base.append(base_m)
        all_split.append(split_m)
    if len(all_x) == 0:
        return None
    return (
        np.concatenate(all_x, axis=0),
        np.concatenate(all_y, axis=0),
        np.concatenate(all_base, axis=0),
        np.concatenate(all_split, axis=0)
    )

def cap_by_split(x, y, base, split, max_train=100000, max_val=30000, max_test=30000, seed=42):
    rng = np.random.default_rng(seed)
    idx_train = np.where(split == "train")[0]
    idx_val = np.where(split == "val")[0]
    idx_test = np.where(split == "test")[0]

    if len(idx_train) > max_train:
        idx_train = rng.choice(idx_train, size=max_train, replace=False)
    if len(idx_val) > max_val:
        idx_val = rng.choice(idx_val, size=max_val, replace=False)
    if len(idx_test) > max_test:
        idx_test = rng.choice(idx_test, size=max_test, replace=False)

    idx = np.concatenate([idx_train, idx_val, idx_test])
    idx.sort()
    return x[idx], y[idx], base[idx], split[idx]

def inverse_from_diff(y_diff, base, d):
    if d == 0:
        return np.asarray(y_diff)
    return np.asarray(base).reshape(-1, 1) + np.cumsum(np.asarray(y_diff), axis=1)

def regression_metrics(y_true, y_pred):
    yt = np.asarray(y_true).reshape(-1)
    yp = np.asarray(y_pred).reshape(-1)
    mae = float(mean_absolute_error(yt, yp))
    rmse_val = float(np.sqrt(mean_squared_error(yt, yp)))
    r2 = float(r2_score(yt, yp))
    mape = float(np.mean(np.abs((yt - yp) / (np.abs(yt) + 1e-6))) * 100.0)
    return {"MAE": mae, "RMSE": rmse_val, "R2": r2, "MAPE(%)": mape}

def build_lstm(window, horizon, u1, u2, dense_units, dropout, lr):
    model = Sequential([
        LSTM(u1, return_sequences=True, input_shape=(window, 1)),
        Dropout(dropout),
        LSTM(u2),
        Dropout(dropout),
        Dense(dense_units, activation="relu"),
        Dense(horizon)
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr), loss="mse")
    return model

In [4]:
global_series = hourly.groupby("hour")[target_col].mean().sort_index()
diff_order, diff_test = choose_diff_order(global_series, max_d=1, alpha=0.05)
print("Differencing order selected:", diff_order)
print("ADF p-values by order:", diff_test)

horizon = 4
windows = [16, 28]
results = []
param_logs = []

for window in windows:
    started = time.time()
    built = build_panel_windows(hourly, target_col, window, horizon, diff_order)
    if built is None:
        continue

    X, y_diff, base, split = built
    X, y_diff, base, split = cap_by_split(X, y_diff, base, split, max_train=100000, max_val=30000, max_test=30000, seed=42 + window)

    tr = np.where(split == "train")[0]
    va = np.where(split == "val")[0]
    te = np.where(split == "test")[0]

    X_train = X[tr]
    y_train = y_diff[tr]
    X_val = X[va]
    y_val = y_diff[va]
    X_test = X[te]
    y_test = y_diff[te]
    base_test = base[te]

    X_train_flat = X_train.reshape(len(X_train), -1)
    X_val_flat = X_val.reshape(len(X_val), -1)
    X_test_flat = X_test.reshape(len(X_test), -1)

    scaler_flat = StandardScaler()
    X_train_sc = scaler_flat.fit_transform(X_train_flat)
    X_val_sc = scaler_flat.transform(X_val_flat)
    X_test_sc = scaler_flat.transform(X_test_flat)

    rng = np.random.default_rng(42 + window)

    def take_idx(n, max_n):
        idx = np.arange(n)
        if n <= max_n:
            return idx
        return rng.choice(idx, size=max_n, replace=False)

    tr_tune = take_idx(len(X_train_sc), 40000)
    va_tune = take_idx(len(X_val_sc), 12000)

    rf_space = {
        "n_estimators": [200, 300, 500],
        "max_depth": [8, 12, 16, None],
        "min_samples_leaf": [1, 2, 4],
        "max_features": ["sqrt", 0.8, 1.0]
    }

    best_rf_params = None
    best_rf_score = np.inf
    for params in ParameterSampler(rf_space, n_iter=8, random_state=42 + window):
        model = RandomForestRegressor(random_state=42, n_jobs=-1, **params)
        model.fit(X_train_sc[tr_tune], y_train[tr_tune])
        pred = model.predict(X_val_sc[va_tune])
        score = rmse_flat(y_val[va_tune], pred)
        if score < best_rf_score:
            best_rf_score = score
            best_rf_params = params

    rf_model = RandomForestRegressor(random_state=42, n_jobs=-1, **best_rf_params)
    rf_model.fit(X_train_sc, y_train)

    et_space = {
        "n_estimators": [200, 300, 500],
        "max_depth": [8, 12, 16, None],
        "min_samples_leaf": [1, 2, 4],
        "max_features": ["sqrt", 0.8, 1.0]
    }

    best_et_params = None
    best_et_score = np.inf
    for params in ParameterSampler(et_space, n_iter=8, random_state=52 + window):
        model = ExtraTreesRegressor(random_state=42, n_jobs=-1, **params)
        model.fit(X_train_sc[tr_tune], y_train[tr_tune])
        pred = model.predict(X_val_sc[va_tune])
        score = rmse_flat(y_val[va_tune], pred)
        if score < best_et_score:
            best_et_score = score
            best_et_params = params

    et_model = ExtraTreesRegressor(random_state=42, n_jobs=-1, **best_et_params)
    et_model.fit(X_train_sc, y_train)

    dnn_space = {
        "hidden_layer_sizes": [(128, 64), (256, 128), (256, 128, 64)],
        "alpha": [1e-4, 1e-3, 1e-2],
        "learning_rate_init": [1e-3, 5e-4],
        "batch_size": [256, 512]
    }

    best_dnn_params = None
    best_dnn_score = np.inf
    for params in ParameterSampler(dnn_space, n_iter=8, random_state=62 + window):
        model = MLPRegressor(
            activation="relu",
            solver="adam",
            max_iter=120,
            early_stopping=True,
            validation_fraction=0.1,
            n_iter_no_change=8,
            random_state=42,
            **params
        )
        model.fit(X_train_sc[tr_tune], y_train[tr_tune])
        pred = model.predict(X_val_sc[va_tune])
        score = rmse_flat(y_val[va_tune], pred)
        if score < best_dnn_score:
            best_dnn_score = score
            best_dnn_params = params

    dnn_model = MLPRegressor(
        activation="relu",
        solver="adam",
        max_iter=160,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=10,
        random_state=42,
        **best_dnn_params
    )
    dnn_model.fit(X_train_sc, y_train)

    x_lstm_scaler = StandardScaler()
    y_lstm_scaler = StandardScaler()

    X_train_lstm_2d = x_lstm_scaler.fit_transform(X_train_flat)
    X_val_lstm_2d = x_lstm_scaler.transform(X_val_flat)
    X_test_lstm_2d = x_lstm_scaler.transform(X_test_flat)

    X_train_lstm = X_train_lstm_2d.reshape(len(X_train_lstm_2d), window, 1)
    X_val_lstm = X_val_lstm_2d.reshape(len(X_val_lstm_2d), window, 1)
    X_test_lstm = X_test_lstm_2d.reshape(len(X_test_lstm_2d), window, 1)

    y_train_lstm = y_lstm_scaler.fit_transform(y_train)
    y_val_lstm = y_lstm_scaler.transform(y_val)

    tr_lstm_tune = take_idx(len(X_train_lstm), 30000)
    va_lstm_tune = take_idx(len(X_val_lstm), 8000)

    lstm_configs = [
        {"u1": 64, "u2": 32, "dense": 32, "drop": 0.2, "lr": 1e-3, "batch": 256},
        {"u1": 96, "u2": 48, "dense": 32, "drop": 0.2, "lr": 1e-3, "batch": 256},
        {"u1": 64, "u2": 32, "dense": 64, "drop": 0.15, "lr": 5e-4, "batch": 256},
        {"u1": 96, "u2": 64, "dense": 32, "drop": 0.25, "lr": 5e-4, "batch": 512}
    ]

    best_lstm_cfg = None
    best_lstm_score = np.inf

    for cfg in lstm_configs:
        model = build_lstm(window, horizon, cfg["u1"], cfg["u2"], cfg["dense"], cfg["drop"], cfg["lr"])
        es = EarlyStopping(patience=3, restore_best_weights=True, monitor="val_loss")
        model.fit(
            X_train_lstm[tr_lstm_tune],
            y_train_lstm[tr_lstm_tune],
            validation_data=(X_val_lstm[va_lstm_tune], y_val_lstm[va_lstm_tune]),
            epochs=20,
            batch_size=cfg["batch"],
            callbacks=[es],
            verbose=0
        )
        pred_sc = model.predict(X_val_lstm[va_lstm_tune], verbose=0)
        pred = y_lstm_scaler.inverse_transform(pred_sc)
        score = rmse_flat(y_val[va_lstm_tune], pred)
        if score < best_lstm_score:
            best_lstm_score = score
            best_lstm_cfg = cfg

    lstm_model = build_lstm(
        window, horizon,
        best_lstm_cfg["u1"],
        best_lstm_cfg["u2"],
        best_lstm_cfg["dense"],
        best_lstm_cfg["drop"],
        best_lstm_cfg["lr"]
    )

    es_final = EarlyStopping(patience=4, restore_best_weights=True, monitor="val_loss")
    lstm_model.fit(
        X_train_lstm,
        y_train_lstm,
        validation_data=(X_val_lstm, y_val_lstm),
        epochs=25,
        batch_size=best_lstm_cfg["batch"],
        callbacks=[es_final],
        verbose=0
    )

    pred_rf_val = rf_model.predict(X_val_sc)
    pred_rf_test = rf_model.predict(X_test_sc)
    pred_et_val = et_model.predict(X_val_sc)
    pred_et_test = et_model.predict(X_test_sc)
    pred_dnn_val = dnn_model.predict(X_val_sc)
    pred_dnn_test = dnn_model.predict(X_test_sc)

    pred_lstm_val_sc = lstm_model.predict(X_val_lstm, verbose=0)
    pred_lstm_test_sc = lstm_model.predict(X_test_lstm, verbose=0)
    pred_lstm_val = y_lstm_scaler.inverse_transform(pred_lstm_val_sc)
    pred_lstm_test = y_lstm_scaler.inverse_transform(pred_lstm_test_sc)

    meta_train_x = np.hstack([pred_rf_val, pred_et_val, pred_dnn_val, pred_lstm_val])
    meta_train_y = y_val
    meta_test_x = np.hstack([pred_rf_test, pred_et_test, pred_dnn_test, pred_lstm_test])

    cut = int(len(meta_train_x) * 0.8)
    if cut < 20:
        cut = max(1, len(meta_train_x) - 1)
    m_tr = np.arange(0, cut)
    m_va = np.arange(cut, len(meta_train_x))

    gb_meta_space = {
        "n_estimators": [200, 300, 500],
        "learning_rate": [0.03, 0.05, 0.1],
        "max_depth": [2, 3, 4],
        "subsample": [0.7, 0.9, 1.0],
        "min_samples_leaf": [1, 2, 5]
    }

    best_meta_params = None
    best_meta_score = np.inf

    for params in ParameterSampler(gb_meta_space, n_iter=8, random_state=72 + window):
        meta_model = MultiOutputRegressor(
            GradientBoostingRegressor(random_state=42, **params),
            n_jobs=-1
        )
        meta_model.fit(meta_train_x[m_tr], meta_train_y[m_tr])
        pred = meta_model.predict(meta_train_x[m_va])
        score = rmse_flat(meta_train_y[m_va], pred)
        if score < best_meta_score:
            best_meta_score = score
            best_meta_params = params

    meta_model = MultiOutputRegressor(
        GradientBoostingRegressor(random_state=42, **best_meta_params),
        n_jobs=-1
    )
    meta_model.fit(meta_train_x, meta_train_y)
    pred_meta_test = meta_model.predict(meta_test_x)

    true_level = inverse_from_diff(y_test, base_test, diff_order)
    pred_map = {
        "RandomForest": pred_rf_test,
        "ExtraTrees": pred_et_test,
        "DNN": pred_dnn_test,
        "LSTM": pred_lstm_test,
        "Stacking_GB": pred_meta_test
    }

    for model_name, pred_diff in pred_map.items():
        pred_level = inverse_from_diff(pred_diff, base_test, diff_order)
        m = regression_metrics(true_level, pred_level)
        m["Window"] = window
        m["Model"] = model_name
        for h in range(horizon):
            m[f"RMSE_t+{h+1}"] = float(np.sqrt(mean_squared_error(true_level[:, h], pred_level[:, h])))
        results.append(m)

    param_logs.append({
        "Window": window,
        "RF_Params": str(best_rf_params),
        "ET_Params": str(best_et_params),
        "DNN_Params": str(best_dnn_params),
        "LSTM_Params": str(best_lstm_cfg),
        "Meta_Params": str(best_meta_params),
        "RunSeconds": round(time.time() - started, 2)
    })

    print("Window", window, "completed in", round(time.time() - started, 2), "seconds")

Differencing order selected: 0
ADF p-values by order: {0: 9.244224881325133e-06}


I0000 00:00:1776796572.311350      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1776796572.317190      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
I0000 00:00:1776796576.468282     364 cuda_dnn.cc:529] Loaded cuDNN version 91002


Window 16 completed in 110.76 seconds
Window 28 completed in 89.33 seconds


In [5]:
results_df = pd.DataFrame(results).sort_values(["Window", "RMSE"]).reset_index(drop=True)
params_df = pd.DataFrame(param_logs)

display(results_df)
display(params_df)

best_per_window = results_df.loc[results_df.groupby("Window")["RMSE"].idxmin()].sort_values("Window")
display(best_per_window)

results_df.to_csv("paper_stacking_gb_metrics.csv", index=False)
params_df.to_csv("paper_stacking_gb_params.csv", index=False)
print("Saved: paper_stacking_gb_metrics.csv, paper_stacking_gb_params.csv")

,MAE,RMSE,R2,MAPE(%),Window,Model,RMSE_t+1,RMSE_t+2,RMSE_t+3,RMSE_t+4
0,294.706395,379.469681,0.295159,165.948054,16,ExtraTrees,300.060001,387.883338,412.238560,406.889218
1,297.501862,385.687941,0.271870,163.936539,16,LSTM,327.358891,402.462343,405.314365,401.996560
2,286.751160,387.605268,0.264613,144.120972,16,DNN,310.533513,397.731576,421.992336,410.185781
3,318.280353,409.927555,0.177471,184.767479,16,RandomForest,311.172927,411.012776,446.688114,454.831877
4,330.317461,439.416582,0.054874,181.715787,16,Stacking_GB,337.614136,442.792075,472.263014,489.149514
5,282.775557,370.003100,0.329888,160.377532,28,ExtraTrees,305.605444,382.765963,396.682395,387.747062
6,300.838142,387.859666,0.263647,176.111154,28,RandomForest,310.760392,395.663542,418.769951,416.233769
7,293.958405,397.984100,0.224703,153.938812,28,DNN,354.910299,406.751671,419.341187,407.811346
8,311.773071,419.751731,0.137574,159.325806,28,LSTM,426.732773,417.667612,411.458572,422.990322
9,376.607364,499.877022,-0.223103,217.954019,28,Stacking_GB,416.454194,524.181063,543.565505,505.811040


,Window,RF_Params,ET_Params,DNN_Params,LSTM_Params,Meta_Params,RunSeconds
0,16,"{'n_estimators': 500, 'min_samples_leaf': 4, '...","{'n_estimators': 200, 'min_samples_leaf': 2, '...","{'learning_rate_init': 0.001, 'hidden_layer_si...","{'u1': 96, 'u2': 48, 'dense': 32, 'drop': 0.2,...","{'subsample': 1.0, 'n_estimators': 300, 'min_s...",110.76
1,28,"{'n_estimators': 500, 'min_samples_leaf': 2, '...","{'n_estimators': 300, 'min_samples_leaf': 4, '...","{'learning_rate_init': 0.001, 'hidden_layer_si...","{'u1': 96, 'u2': 48, 'dense': 32, 'drop': 0.2,...","{'subsample': 1.0, 'n_estimators': 500, 'min_s...",89.33


,MAE,RMSE,R2,MAPE(%),Window,Model,RMSE_t+1,RMSE_t+2,RMSE_t+3,RMSE_t+4
0,294.706395,379.469681,0.295159,165.948054,16,ExtraTrees,300.060001,387.883338,412.238560,406.889218
5,282.775557,370.003100,0.329888,160.377532,28,ExtraTrees,305.605444,382.765963,396.682395,387.747062


Saved: paper_stacking_gb_metrics.csv, paper_stacking_gb_params.csv
